# DermaFuzzy-XAI: Explainable Melanoma Diagnosis on HAM10000
**Production Q1 Research Pipeline** ? Enable `Runtime -> Change runtime type -> T4 GPU`, then click `Runtime -> Run all`.

In [ ]:
# Cell 1: Clone clean repository & install dependencies
%cd /content
!rm -rf dermafuzzy-xai
!git clone https://github.com/A11-HELAL/dermafuzzy-xai.git
%cd /content/dermafuzzy-xai
!git log -n 1 --oneline
!pip install -q -e . kagglehub pandas scikit-learn


In [ ]:
# Cell 2: Download HAM10000 benchmark dataset via KaggleHub
import os, kagglehub

dataset_path = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
os.environ['HAM_DIR'] = dataset_path
os.environ['HAM_CSV'] = os.path.join(dataset_path, 'HAM10000_metadata.csv')
print('Dataset ready at:', os.environ['HAM_DIR'])
print('Metadata exists:', os.path.exists(os.environ['HAM_CSV']))


In [ ]:
# Cell 3: Run Real 5-Stage HAM10000 Training, TTA Grad-CAM & 3-Tier Ablation Study
!python services/colab_trainer/train_ham10000.py \
  --data-dir "$HAM_DIR" \
  --metadata-csv "$HAM_CSV" \
  --output-dir artifacts \
  --epochs 15 \
  --batch-size 32


In [ ]:
# Cell 4: Display Real 3-Tier Ablation Table & 4-Panel Clinical XAI Report (300 DPI)
from IPython.display import Image, Markdown, display

with open('artifacts/ablation_report.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

display(Image('artifacts/sample_4panel_xai.png'))


In [ ]:
# Cell 5: Download Complete Paper Artifacts & Trained Weights (ZIP)
from google.colab import files
!zip -r dermafuzzy_real_ham10000_artifacts.zip artifacts/
files.download('dermafuzzy_real_ham10000_artifacts.zip')


In [ ]:
# Cell 6: Run Empirical Artifact Perturbation Study & Statistical Analysis (DeLong, McNemar, Bootstrap 95% CIs)
!python services/colab_trainer/artifact_perturbation_study.py \
  --data-dir "$HAM_DIR" \
  --metadata-csv "$HAM_CSV" \
  --model-path artifacts/mobilenetv2_ham10000.keras \
  --calibration artifacts/fis_calibration.json \
  --out-dir artifacts/artifact_study \
  --batch-size 32

!python services/colab_trainer/stats_analysis.py \
  --cases-csv artifacts/artifact_study/artifact_study_cases.csv \
  --out-dir artifacts/stats \
  --n-boot 2000 \
  --seed 42 \
  --threshold 0.50


In [ ]:
# Cell 7: Run Leak-Free 5-Fold Lesion-Grouped Cross-Validation (with Checkpoint Resume)
!python services/colab_trainer/run_5fold_cv.py \
  --data-dir "$HAM_DIR" \
  --metadata-csv "$HAM_CSV" \
  --output-dir artifacts/cv_5fold \
  --epochs 20 \
  --batch-size 32 \
  --resume


In [ ]:
# Cell 8: Render All 4 Q1 300-DPI Figures, Display Reports, and Export Complete v2.0 Artifacts ZIP
from IPython.display import Image, Markdown, display
from google.colab import files

!python services/colab_trainer/plot_q1_figures.py \
  --cases artifacts/artifact_study/artifact_study_cases.csv \
  --stats artifacts/stats/stats_summary.json \
  --calibration artifacts/fis_calibration.json \
  --cv-summary artifacts/cv_5fold/cv_summary.json \
  --out-dir artifacts/q1_figs

for report_path in ('artifacts/stats/stats_report.md', 'artifacts/cv_5fold/cv_report.md'):
    with open(report_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))

for fig_name in (
    'fig_roc_pr.png',
    'fig_threshold_triage.png',
    'fig_gate_and_memberships.png',
    'fig_artifact_resilience.png',
):
    display(Image(f'artifacts/q1_figs/{fig_name}'))

!zip -r dermafuzzy_v2_bspc_artifacts.zip artifacts/
files.download('dermafuzzy_v2_bspc_artifacts.zip')
